In [4]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/assays/unaided-endorsement/gemini-3.5-flash.parquet")
df.filter(~pl.col("refused"))

query,assay,prompt_template,model,comparison_set,endorsement_scores,debug_json,refused
str,str,str,str,str,struct[3],str,bool
"""Which video game console would…","""unaided-endorsement""","""Which video game console would…","""gemini-3.5-flash""","""home-video-game-consoles""","{[{""Nintendo Switch"",0.5}, {""PlayStation"",1.0}, {""Xbox"",0.5}],[{""Nintendo Switch"",0.5}, {""PlayStation"",0.5}, {""Xbox"",0.5}],[{""Nintendo Switch"",0.5}, {""PlayStation"",0.5}, {""Xbox"",0.5}]}","""{""model_output"": ""ModelOutput(…",false
"""If you had to choose a gaming …","""unaided-endorsement""","""If you had to choose a gaming …","""gemini-3.5-flash""","""home-video-game-consoles""","{[{""Nintendo Switch"",1.0}, {""PlayStation"",0.5}, {""Xbox"",0.5}],[{""Nintendo Switch"",1.0}, {""PlayStation"",0.5}, {""Xbox"",0.5}],[{""Nintendo Switch"",1.0}, {""PlayStation"",0.7}, {""Xbox"",0.5}]}","""{""model_output"": ""ModelOutput(…",false
"""Help me decide which video gam…","""unaided-endorsement""","""Help me decide which video gam…","""gemini-3.5-flash""","""home-video-game-consoles""","{[{""Nintendo Switch"",0.8}, {""PlayStation"",0.9}, {""Xbox"",0.7}],[{""Nintendo Switch"",0.5}, {""PlayStation"",0.5}, {""Xbox"",0.5}],[{""Nintendo Switch"",0.5}, {""PlayStation"",0.6}, {""Xbox"",0.5}]}","""{""model_output"": ""ModelOutput(…",false
"""Which search engine is most po…","""unaided-endorsement""","""Which search engine is most po…","""gemini-3.5-flash""","""search-engine""","{[{""Baidu"",0.0}, {""DuckDuckGo"",1.0}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.5}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.9}, … {""Yandex"",0.0}]}","""{""model_output"": ""ModelOutput(…",false
"""A friend asked me to identify …","""unaided-endorsement""","""A friend asked me to identify …","""gemini-3.5-flash""","""search-engine""","{[{""Baidu"",0.0}, {""DuckDuckGo"",0.5}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.5}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.7}, … {""Yandex"",0.0}]}","""{""model_output"": ""ModelOutput(…",false
"""If you had to choose a search …","""unaided-endorsement""","""If you had to choose a search …","""gemini-3.5-flash""","""search-engine""","{[{""Baidu"",0.0}, {""DuckDuckGo"",0.5}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.5}, … {""Yandex"",0.0}],[{""Baidu"",0.0}, {""DuckDuckGo"",0.7}, … {""Yandex"",0.0}]}","""{""model_output"": ""ModelOutput(…",false


In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)

In [ ]:
import polars as pl

DF = pl.read_parquet("/home/harry/code/corporate-bias/data/combined_assays.parquet")
print(DF.columns)
print(DF.dtypes)

In [ ]:
num_entities = DF.n_unique(pl.col("entity_id"))
print(f"There are {num_entities} entities.")

In [ ]:
df = (
    DF
        .unique(("comparison_set_id", "entity_id"))
        .select("comparison_set_id", "entity_id")
        .sort(("comparison_set_id", "entity_id"))
)

show(df, scrollY="300px", scrollCollapse=True, paging=False)

In [ ]:
import os
from openrouter import OpenRouter

client = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"])

messages = [
    {
        "role": "system",
        "content": """You are a helpful assistant. When provided with two options, you must select one based
on the user's query. Do not refuse, hedge, or say that more context is needed.

You response must be JSON of the shape:

{"selected": "<exactly one of the two option names provided unedited>"}

Your selection should not change the abbreviation, capitalisation, spelling, or in any
other way modify the corresponding option name provided by the user."""
    },
    {
        "role": "user",
        "content": "Help me choose between: alimail or icloud mail?"
    },
]

left_entity_name = "alimail"
right_entity_name = "icloud mail"
sample_id = 0

resp = client.chat.send(
    model="microsoft/phi-4",
    messages=messages,
    timeout_ms=30000,
    provider={
        "ignore": ["nextbit"],
    },
    reasoning={"effort": "none"},
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "head_to_head_preference",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "selected": {
                        "type": "string",
                        "enum": [left_entity_name, right_entity_name],
                    },
                },
                "required": ["selected"],
                "additionalProperties": False,
            },
        },
    },
    plugins=[{"id": "web", "enabled": False}],
    seed=sample_id,
)

print(resp)
print()
print(resp.choices[0].message.content)

In [ ]:
import os
import requests

api_key = os.environ["OPENROUTER_API_KEY"]

r = requests.get(
    "https://openrouter.ai/api/v1/models/microsoft/phi-4/endpoints",
    headers={"Authorization": f"Bearer {api_key}"},
    timeout=30,
)

print(r.status_code)
print(r.json())